# GenMoji · FLUX 3 LoRA training on Colab

Train a text-to-image OpenMoji style adapter for **FLUX 3 Image Klein Base**. In Colab, use **Runtime → Change runtime type** and select a GPU with native BF16 support. An A100 is the recommended starting point. An L4 can run BF16, but available memory may limit training. The BF16 recipe requires a different runtime if Colab assigns a T4.

The default is a **20-step smoke test** on 64 training images. Run the cells in order. The notebook can reuse the `GenMoji/openmoji-512-90-10.tar.gz` archive from the diffusion notebook. It also accepts the prepared dataset directory or an uploaded archive.

[Diffusers FLUX 3 trainer](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/README_flux3_klein.md) · [Base model](https://huggingface.co/black-forest-labs/flux-3-image-klein-base) · [Colab GPU options](https://colab.research.google.com/notebooks/pro.ipynb)

## 1. Settings and GPU check

Set `MOUNT_DRIVE = False` to use Colab's temporary storage. With Drive mounted, outputs and checkpoints go under `MyDrive/GenMoji/flux3-lora`. Set `DATA_SOURCE` if your archive or prepared directory is stored elsewhere.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Use None to look for the default Drive archive or an existing Colab dataset.
DATA_SOURCE = None
MOUNT_DRIVE = True
RUN_NAME = "flux3-openmoji-90-10-smoke"
TRAIN_IMAGE_LIMIT = 64  # Set to None to use the full training split.
MAX_TRAIN_STEPS = 20
CHECKPOINT_EVERY = 20
RESUME_LATEST = False
SEED = 42

MODEL_ID = "black-forest-labs/flux-3-image-klein-base"
STYLE_PROMPT = "OpenMoji style emoji"
WORK_DIR = Path("/content/genmoji-flux3")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Query in a subprocess so the notebook kernel leaves GPU memory to training.
subprocess.run([sys.executable, "-c", (
    "import torch; "
    "assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'; "
    "p = torch.cuda.get_device_properties(0); "
    "print(f'{p.name} | {p.total_memory / (1024 ** 3):.1f} GiB VRAM', flush=True); "
    "assert torch.cuda.is_bf16_supported(including_emulation=False), "
    "'This recipe requires native BF16. Select an A100 or L4 runtime.'"
)], check=True)

## 2. Load the prepared OpenMoji training split

With Drive mounted, the cell looks for `MyDrive/GenMoji/openmoji-512-90-10.tar.gz`. Otherwise it looks for a prepared dataset in your Colab checkout and prompts for an archive if needed. To make the archive from the GenMoji root on your Mac, run

```bash
COPYFILE_DISABLE=1 tar -czf openmoji-512-90-10.tar.gz -C data/processed openmoji-512-90-10
```

Each image keeps its original `text` caption from `metadata.jsonl`. The default archive uses the shared 90/10 train/validation split. Only the training folder is passed to the trainer. OpenMoji artwork is licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

In [ ]:
import json
import random
import shutil
import tarfile
import tempfile

from google.colab import drive, files

drive_root = Path("/content/drive/MyDrive/GenMoji")
if MOUNT_DRIVE:
    drive.mount("/content/drive")
output_root = drive_root / "flux3-lora" if MOUNT_DRIVE else WORK_DIR / "outputs"
output_dir = output_root / RUN_NAME

candidates = [Path("/content/GenMoji/data/processed/openmoji-512-90-10")]
if MOUNT_DRIVE:
    candidates.insert(0, drive_root / "openmoji-512-90-10.tar.gz")
if DATA_SOURCE:
    source = Path(DATA_SOURCE).expanduser()
    if not source.exists():
        raise FileNotFoundError(source)
else:
    source = next((path for path in candidates if path.exists()), None)
    if source is None:
        print("Upload the prepared openmoji-512-90-10.tar.gz archive.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload one dataset archive, or set DATA_SOURCE and rerun.")
        source = Path(next(iter(uploaded))).resolve()
        del uploaded

def find_train_dir(root):
    candidates = (
        root / "openmoji-512-90-10/imagefolder/train",
        root / "openmoji-512/imagefolder/train",
        root / "imagefolder/train",
        root / "train",
        root,
    )
    for candidate in candidates:
        if candidate.name == "train" and (candidate / "metadata.jsonl").is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Expected imagefolder/train/metadata.jsonl under {root}.")

if source.is_dir():
    train_dir = find_train_dir(source)
else:
    archive_dir = Path(tempfile.mkdtemp(prefix="openmoji-", dir=WORK_DIR))
    with tarfile.open(source, "r:*") as archive:
        archive.extractall(archive_dir, filter="data")
    train_dir = find_train_dir(archive_dir)

metadata = train_dir / "metadata.jsonl"
rows = [json.loads(line) for line in metadata.read_text(encoding="utf-8").splitlines() if line.strip()]
if not rows:
    raise ValueError(f"No training captions in {metadata}.")

if TRAIN_IMAGE_LIMIT is not None:
    if TRAIN_IMAGE_LIMIT < 1:
        raise ValueError("TRAIN_IMAGE_LIMIT must be positive or None.")
    rows = random.Random(SEED).sample(rows, min(TRAIN_IMAGE_LIMIT, len(rows)))
    trainer_data = Path(tempfile.mkdtemp(prefix="train-subset-", dir=WORK_DIR))
else:
    trainer_data = train_dir

for row in rows:
    name = row["file_name"]
    if Path(name).name != name or not name.lower().endswith(".png"):
        raise ValueError(f"Expected a PNG filename in {metadata}, got {name!r}.")
    if not isinstance(row["text"], str) or not row["text"].strip():
        raise ValueError(f"Missing caption for {name}.")
    image = train_dir / name
    if not image.is_file():
        raise FileNotFoundError(image)
    if trainer_data != train_dir:
        shutil.copyfile(image, trainer_data / name)

if trainer_data != train_dir:
    contents = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)
    (trainer_data / "metadata.jsonl").write_text(contents, encoding="utf-8")

print(f"Using {len(rows):,} captioned images from {trainer_data}")
print(f"Example caption: {rows[0]['text']}")
print(f"Output directory: {output_dir}")

## 3. Install the FLUX 3 trainer

This installs Diffusers from the same checkout as its training script. The cell prints the checkout commit and checks the trainer's CLI before downloading model weights.

In [ ]:
diffusers_dir = WORK_DIR / "diffusers"
if not (diffusers_dir / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/huggingface/diffusers.git", str(diffusers_dir),
    ], check=True)

trainer_dir = diffusers_dir / "examples/dreambooth"
trainer = trainer_dir / "train_dreambooth_lora_flux3_klein.py"
requirements = trainer_dir / "requirements_flux3_klein.txt"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(diffusers_dir), "-r", str(requirements),
], check=True)
diffusers_commit = subprocess.check_output(
    ["git", "-C", str(diffusers_dir), "rev-parse", "HEAD"], text=True,
).strip()
print(f"Diffusers commit: {diffusers_commit}")
subprocess.run([sys.executable, str(trainer), "--help"], stdout=subprocess.DEVNULL, check=True)
print("FLUX 3 trainer imports successfully.")

## 4. Train the adapter

The first run downloads the Base model. The trainer then caches image latents and caption embeddings before it shows training steps. If GPU memory runs out after switching to the full dataset, use an A100 or return to a smaller subset.

For a longer first experiment, set `TRAIN_IMAGE_LIMIT = None`, `MAX_TRAIN_STEPS = 1000`, `CHECKPOINT_EVERY = 250`, and a new `RUN_NAME` in step 1, then rerun steps 1–4. To resume an interrupted run, keep its name, dataset, and training settings, set `RESUME_LATEST = True`, and rerun. Steps count from the beginning of the run.

In [ ]:
import shlex

adapter_path = output_dir / "pytorch_lora_weights.safetensors"
if RESUME_LATEST:
    if not any(output_dir.glob("checkpoint-*")):
        raise FileNotFoundError(f"No checkpoints to resume in {output_dir}.")
elif output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(f"{output_dir} already contains a run. Choose another RUN_NAME or resume.")

output_dir.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "1", "--num_machines", "1", "--mixed_precision", "bf16",
    str(trainer),
    "--pretrained_model_name_or_path", MODEL_ID,
    "--dataset_name", str(trainer_data),
    "--image_column", "image",
    "--caption_column", "text",
    "--instance_prompt", STYLE_PROMPT,
    "--output_dir", str(output_dir),
    "--mixed_precision", "bf16",
    "--resolution", "512",
    "--train_batch_size", "1",
    "--gradient_checkpointing", "--cache_latents", "--offload",
    "--rank", "16", "--lora_alpha", "16",
    "--learning_rate", "1e-4", "--lr_scheduler", "constant", "--lr_warmup_steps", "0",
    "--max_train_steps", str(MAX_TRAIN_STEPS),
    "--checkpointing_steps", str(CHECKPOINT_EVERY), "--checkpoints_total_limit", "2",
    "--report_to", "tensorboard",
    "--seed", str(SEED),
]
if RESUME_LATEST:
    command.extend(["--resume_from_checkpoint", "latest"])

print(shlex.join(command), flush=True)
subprocess.run(command, cwd=WORK_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"}, check=True)
if not adapter_path.is_file():
    raise FileNotFoundError(f"Expected the trainer to save {adapter_path}.")
print(f"Adapter saved to {adapter_path}")

## 5. Download the LoRA

With Drive mounted, the file is already saved there. Run the cell below for a browser download. Load it with **FLUX 3 Image Klein Base** and try a prompt beginning with `OpenMoji style emoji of`. [MFLUX adapter loading](https://github.com/mflux-community/mflux/blob/main/src/mflux/models/flux3/README.md#lora)

In [ ]:
adapter_path = output_dir / "pytorch_lora_weights.safetensors"
if not adapter_path.is_file():
    raise FileNotFoundError(f"Run the training cell first. Expected {adapter_path}.")
files.download(str(adapter_path))